# Imports

In [1]:
import pandas as pd
import numpy as np
import os

# File paths

In [2]:
mit_path = "../data/Raw/countypres_2000-2024.csv"

age_path = "../data/Raw/demographic/ACSDT5Y2020.B01001-Data.csv"
race_path = "../data/Raw/demographic/ACSDT5Y2020.B02001-Data.csv"
education_path = "../data/Raw/demographic/ACSDT5Y2020.B15003-Data.csv"

income_path = "../data/Raw/socioeconomic/ACSDT5Y2020.B19013-Data.csv"
poverty_path = "../data/Raw/socioeconomic/ACSDT5Y2020.B17001-Data.csv"
employment_path = "../data/Raw/socioeconomic/ACSDT5Y2020.B23025-Data.csv"
housing_path = "../data/Raw/socioeconomic/ACSDT5Y2020.B25001-Data.csv"

# Load Datasets

In [3]:
mit_df = pd.read_csv(mit_path)

age_df = pd.read_csv(age_path)
race_df = pd.read_csv(race_path)
education_df = pd.read_csv(education_path)

income_df = pd.read_csv(income_path)
poverty_df = pd.read_csv(poverty_path)
employment_df = pd.read_csv(employment_path)
housing_df = pd.read_csv(housing_path)

# Check shapes

In [4]:
print("MIT:", mit_df.shape)
print("Age:", age_df.shape)
print("Race:", race_df.shape)
print("Education:", education_df.shape)
print("Income:", income_df.shape)
print("Poverty:", poverty_df.shape)
print("Employment:", employment_df.shape)
print("Housing:", housing_df.shape)

MIT: (94151, 12)
Age: (3222, 101)
Race: (3222, 23)
Education: (3222, 53)
Income: (3222, 5)
Poverty: (3222, 121)
Employment: (3222, 17)
Housing: (3222, 5)


# Preview one ACS dataset and MIT dataset

In [5]:
print("MIT Dataset")
display(mit_df.head())

print("Age Dataset")
display(age_df.head())

MIT Dataset


,state,county_name,year,state_po,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,OTHER,OTHER,293.0,28281,20260225,TOTAL
1,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,CHASE OLIVER,LIBERTARIAN,65.0,28281,20260225,TOTAL
2,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,KAMALA D HARRIS,DEMOCRAT,7439.0,28281,20260225,TOTAL
3,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,20484.0,28281,20260225,TOTAL
4,ALABAMA,BALDWIN,2024,AL,1003.0,US PRESIDENT,OTHER,OTHER,1276.0,122249,20260225,TOTAL


Age Dataset


,GEO_ID,NAME,B01001_001E,B01001_001M,B01001_002E,B01001_002M,B01001_003E,B01001_003M,B01001_004E,B01001_004M,...,B01001_045M,B01001_046E,B01001_046M,B01001_047E,B01001_047M,B01001_048E,B01001_048M,B01001_049E,B01001_049M,Unnamed: 100
0,Geography,Geographic Area Name,Estimate!!Total:,Margin of Error!!Total:,Estimate!!Total:!!Male:,Margin of Error!!Total:!!Male:,Estimate!!Total:!!Male:!!Under 5 years,Margin of Error!!Total:!!Male:!!Under 5 years,Estimate!!Total:!!Male:!!5 to 9 years,Margin of Error!!Total:!!Male:!!5 to 9 years,...,Margin of Error!!Total:!!Female:!!67 to 69 years,Estimate!!Total:!!Female:!!70 to 74 years,Margin of Error!!Total:!!Female:!!70 to 74 years,Estimate!!Total:!!Female:!!75 to 79 years,Margin of Error!!Total:!!Female:!!75 to 79 years,Estimate!!Total:!!Female:!!80 to 84 years,Margin of Error!!Total:!!Female:!!80 to 84 years,Estimate!!Total:!!Female:!!85 years and over,Margin of Error!!Total:!!Female:!!85 years and...,NaN
1,0500000US01001,"Autauga County, Alabama",55639,*****,27052,167,1727,78,2156,281,...,184,1414,234,1066,229,358,118,637,216,NaN
2,0500000US01003,"Baldwin County, Alabama",218289,*****,105889,253,6082,72,5197,524,...,497,7008,561,4669,489,2556,377,2588,434,NaN
3,0500000US01005,"Barbour County, Alabama",25026,*****,13156,86,660,29,741,137,...,98,766,108,474,112,335,97,341,97,NaN
4,0500000US01007,"Bibb County, Alabama",22374,*****,12022,170,626,89,693,220,...,182,580,190,451,125,272,132,224,112,NaN


# Create ACS cleaning function

In [8]:
def clean_acs_data(df):
    df = df.iloc[1:].copy()
    
    df["county_fips"] = df["GEO_ID"].astype(str).str[-5:]
    
    keep_cols = ["GEO_ID", "NAME", "county_fips"]
    estimate_cols = [col for col in df.columns if str(col).endswith("E")]
    
    # remove duplicate column names while preserving order
    estimate_cols = list(dict.fromkeys(estimate_cols))
    
    df = df[keep_cols + estimate_cols].copy()
    
    # convert only columns that are true Series, not duplicated DataFrames
    for col in estimate_cols:
        if col in df.columns:
            if isinstance(df[col], pd.Series):
                df[col] = pd.to_numeric(df[col], errors="coerce")
    
    return df

# Clean all ACS datasets

In [9]:
age_clean = clean_acs_data(age_df)
race_clean = clean_acs_data(race_df)
education_clean = clean_acs_data(education_df)

income_clean = clean_acs_data(income_df)
poverty_clean = clean_acs_data(poverty_df)
employment_clean = clean_acs_data(employment_df)
housing_clean = clean_acs_data(housing_df)

# Check cleaned ACS data

In [10]:
print("Age cleaned:", age_clean.shape)
display(age_clean.head())

print("Race cleaned:", race_clean.shape)
display(race_clean.head())

Age cleaned: (3221, 53)


,GEO_ID,NAME,county_fips,NAME,B01001_001E,B01001_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,...,B01001_040E,B01001_041E,B01001_042E,B01001_043E,B01001_044E,B01001_045E,B01001_046E,B01001_047E,B01001_048E,B01001_049E
1,0500000US01001,"Autauga County, Alabama",01001,"Autauga County, Alabama",55639,27052,1727,2156,1674,1208,...,1874,2080,835,800,496,836,1414,1066,358,637
2,0500000US01003,"Baldwin County, Alabama",01003,"Baldwin County, Alabama",218289,105889,6082,5197,8289,4305,...,7396,7802,3702,5123,3058,4236,7008,4669,2556,2588
3,0500000US01005,"Barbour County, Alabama",01005,"Barbour County, Alabama",25026,13156,660,741,721,467,...,755,782,360,459,222,566,766,474,335,341
4,0500000US01007,"Bibb County, Alabama",01007,"Bibb County, Alabama",22374,12022,626,693,621,569,...,627,649,286,444,71,461,580,451,272,224
5,0500000US01009,"Blount County, Alabama",01009,"Blount County, Alabama",57755,28677,1844,1738,2034,1257,...,1844,1767,893,1218,708,1050,1415,868,856,850


Race cleaned: (3221, 14)


,GEO_ID,NAME,county_fips,NAME,B02001_001E,B02001_002E,B02001_003E,B02001_004E,B02001_005E,B02001_006E,B02001_007E,B02001_008E,B02001_009E,B02001_010E
1,0500000US01001,"Autauga County, Alabama",01001,"Autauga County, Alabama",55639,42150,10866,155,649,24,374,1421,346,1075
2,0500000US01003,"Baldwin County, Alabama",01003,"Baldwin County, Alabama",218289,186504,19153,1514,2033,10,3398,5677,1519,4158
3,0500000US01005,"Barbour County, Alabama",01005,"Barbour County, Alabama",25026,11587,11929,88,122,1,777,522,232,290
4,0500000US01007,"Bibb County, Alabama",01007,"Bibb County, Alabama",22374,17138,5045,12,56,0,9,114,0,114
5,0500000US01009,"Blount County, Alabama",01009,"Blount County, Alabama",57755,54271,808,55,236,55,1037,1293,264,1029


# Check duplicate FIPS in ACS tables

In [11]:
print("Age duplicate FIPS:", age_clean["county_fips"].duplicated().sum())
print("Race duplicate FIPS:", race_clean["county_fips"].duplicated().sum())
print("Education duplicate FIPS:", education_clean["county_fips"].duplicated().sum())
print("Income duplicate FIPS:", income_clean["county_fips"].duplicated().sum())
print("Poverty duplicate FIPS:", poverty_clean["county_fips"].duplicated().sum())
print("Employment duplicate FIPS:", employment_clean["county_fips"].duplicated().sum())
print("Housing duplicate FIPS:", housing_clean["county_fips"].duplicated().sum())

Age duplicate FIPS: 0
Race duplicate FIPS: 0
Education duplicate FIPS: 0
Income duplicate FIPS: 0
Poverty duplicate FIPS: 0
Employment duplicate FIPS: 0
Housing duplicate FIPS: 0


# Prepare MIT election dataset

In [31]:
mit_2020 = mit_df[mit_df["year"] == 2020].copy()

mit_2020 = mit_2020[
    mit_2020["party"].isin(["DEMOCRAT", "REPUBLICAN"])
].copy()

# convert county_fips safely
mit_2020["county_fips"] = pd.to_numeric(
    mit_2020["county_fips"],
    errors="coerce"
)

# remove missing FIPS rows
mit_2020 = mit_2020[
    mit_2020["county_fips"].notna()
].copy()

# convert to proper 5-digit strings
mit_2020["county_fips"] = (
    mit_2020["county_fips"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

# Check MIT filtered data

In [32]:
print("MIT county_fips sample after cleaning:")
print(mit_2020["county_fips"].head(10).tolist())

print("\nMIT county_fips dtype:")
print(mit_2020["county_fips"].dtype)

print("\nMissing MIT county_fips:")
print(mit_2020["county_fips"].isna().sum())

print("\nMIT dataset shape:")
print(mit_2020.shape)

MIT county_fips sample after cleaning:
['01001', '01001', '01003', '01003', '01005', '01005', '01007', '01007', '01009', '01009']

MIT county_fips dtype:
object

Missing MIT county_fips:
0

MIT dataset shape:
(10232, 12)


# Pivot MIT data to county level

In [33]:
mit_pivot = mit_2020.pivot_table(
    index=[
        "county_fips",
        "state",
        "county_name",
        "totalvotes"
    ],
    columns="party",
    values="candidatevotes",
    aggfunc="sum"
).reset_index()

# Clean pivoted election data

In [34]:
mit_pivot.columns.name = None

mit_pivot = mit_pivot.rename(columns={
    "DEMOCRAT": "democrat_votes",
    "REPUBLICAN": "republican_votes"
})

mit_pivot["democrat_votes"] = (
    mit_pivot["democrat_votes"].fillna(0)
)

mit_pivot["republican_votes"] = (
    mit_pivot["republican_votes"].fillna(0)
)

# Create target variables

In [35]:
mit_pivot["party_winner"] = np.where(
    mit_pivot["democrat_votes"] >
    mit_pivot["republican_votes"],
    1,
    0
)

mit_pivot["dem_vote_share"] = (
    mit_pivot["democrat_votes"] /
    mit_pivot["totalvotes"]
)

# Check election target dataset

In [36]:
print("MIT pivot shape:")
print(mit_pivot.shape)

display(mit_pivot.head())

print("\nParty winner counts:")
print(mit_pivot["party_winner"].value_counts())

print("\nMIT county_fips sample:")
print(mit_pivot["county_fips"].head(10).tolist())

MIT pivot shape:
(3154, 8)


,county_fips,state,county_name,totalvotes,democrat_votes,republican_votes,party_winner,dem_vote_share
0,01001,ALABAMA,AUTAUGA,27770,7503.0,19838.0,0,0.270184
1,01003,ALABAMA,BALDWIN,109679,24578.0,83544.0,0,0.224090
2,01005,ALABAMA,BARBOUR,10518,4816.0,5622.0,0,0.457882
3,01007,ALABAMA,BIBB,9595,1986.0,7525.0,0,0.206983
4,01009,ALABAMA,BLOUNT,27588,2640.0,24711.0,0,0.095694



Party winner counts:
party_winner
0    2596
1     558
Name: count, dtype: int64

MIT county_fips sample:
['01001', '01003', '01005', '01007', '01009', '01011', '01013', '01015', '01017', '01019']


# Keep only one NAME column in ACS merges

In [37]:
race_clean_small = race_clean.drop(columns=["GEO_ID", "NAME"])
education_clean_small = education_clean.drop(columns=["GEO_ID", "NAME"])

income_clean_small = income_clean.drop(columns=["GEO_ID", "NAME"])
poverty_clean_small = poverty_clean.drop(columns=["GEO_ID", "NAME"])
employment_clean_small = employment_clean.drop(columns=["GEO_ID", "NAME"])
housing_clean_small = housing_clean.drop(columns=["GEO_ID", "NAME"])

# Merge all ACS datasets together

In [38]:
acs_merged = age_clean.merge(race_clean_small, on="county_fips", how="inner")
acs_merged = acs_merged.merge(education_clean_small, on="county_fips", how="inner")
acs_merged = acs_merged.merge(income_clean_small, on="county_fips", how="inner")
acs_merged = acs_merged.merge(poverty_clean_small, on="county_fips", how="inner")
acs_merged = acs_merged.merge(employment_clean_small, on="county_fips", how="inner")
acs_merged = acs_merged.merge(housing_clean_small, on="county_fips", how="inner")

# Check merged ACS dataset

In [39]:
print("ACS merged shape:", acs_merged.shape)
display(acs_merged.head())

ACS merged shape: (3221, 156)


,GEO_ID,NAME,county_fips,NAME,B01001_001E,B01001_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,...,B17001_058E,B17001_059E,B23025_001E,B23025_002E,B23025_003E,B23025_004E,B23025_005E,B23025_006E,B23025_007E,B25001_001E
0,0500000US01001,"Autauga County, Alabama",01001,"Autauga County, Alabama",55639,27052,1727,2156,1674,1208,...,2460,1809,44360,26025,25316,24580,736,709,18335,23697
1,0500000US01003,"Baldwin County, Alabama",01003,"Baldwin County, Alabama",218289,105889,6082,5197,8289,4305,...,13017,8410,176852,103116,102795,98768,4027,321,73736,116747
2,0500000US01005,"Barbour County, Alabama",01005,"Barbour County, Alabama",25026,13156,660,741,721,467,...,1279,766,20407,9356,9356,8707,649,0,11051,12057
3,0500000US01007,"Bibb County, Alabama",01007,"Bibb County, Alabama",22374,12022,626,693,621,569,...,957,820,18421,8970,8970,8303,667,0,9451,9237
4,0500000US01009,"Blount County, Alabama",01009,"Blount County, Alabama",57755,28677,1844,1738,2034,1257,...,2668,1914,46083,24133,24089,22836,1253,44,21950,24404


In [40]:
acs_merged["county_fips"] = (
    acs_merged["county_fips"]
    .astype(str)
    .str.strip()
    .str.zfill(5)
)

mit_pivot["county_fips"] = (
    mit_pivot["county_fips"]
    .astype(str)
    .str.strip()
    .str.zfill(5)
)

acs_fips = set(acs_merged["county_fips"])
mit_fips = set(mit_pivot["county_fips"])

common_fips = acs_fips.intersection(mit_fips)

print("ACS FIPS count:", len(acs_fips))
print("MIT FIPS count:", len(mit_fips))
print("Common FIPS count:", len(common_fips))

print("\nSample common FIPS:")
print(list(common_fips)[:10])

ACS FIPS count: 3221
MIT FIPS count: 3154
Common FIPS count: 3115

Sample common FIPS:
['28059', '26143', '20089', '39083', '13257', '06039', '28031', '39095', '02020', '54029']


# Merge ACS with election targets

In [41]:
final_df = acs_merged.merge(
    mit_pivot,
    on="county_fips",
    how="inner"
)

print("Final merged dataset shape:")
print(final_df.shape)

display(final_df.head())

Final merged dataset shape:
(3115, 163)


,GEO_ID,NAME,county_fips,NAME,B01001_001E,B01001_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,...,B23025_006E,B23025_007E,B25001_001E,state,county_name,totalvotes,democrat_votes,republican_votes,party_winner,dem_vote_share
0,0500000US01001,"Autauga County, Alabama",01001,"Autauga County, Alabama",55639,27052,1727,2156,1674,1208,...,709,18335,23697,ALABAMA,AUTAUGA,27770,7503.0,19838.0,0,0.270184
1,0500000US01003,"Baldwin County, Alabama",01003,"Baldwin County, Alabama",218289,105889,6082,5197,8289,4305,...,321,73736,116747,ALABAMA,BALDWIN,109679,24578.0,83544.0,0,0.224090
2,0500000US01005,"Barbour County, Alabama",01005,"Barbour County, Alabama",25026,13156,660,741,721,467,...,0,11051,12057,ALABAMA,BARBOUR,10518,4816.0,5622.0,0,0.457882
3,0500000US01007,"Bibb County, Alabama",01007,"Bibb County, Alabama",22374,12022,626,693,621,569,...,0,9451,9237,ALABAMA,BIBB,9595,1986.0,7525.0,0,0.206983
4,0500000US01009,"Blount County, Alabama",01009,"Blount County, Alabama",57755,28677,1844,1738,2034,1257,...,44,21950,24404,ALABAMA,BLOUNT,27588,2640.0,24711.0,0,0.095694


# Check final merged dataset

In [42]:
print("Missing values summary:")

print(
    final_df.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

Missing values summary:
B19013_001E    1
NAME           0
GEO_ID         0
NAME           0
B01001_001E    0
B01001_002E    0
B01001_003E    0
B01001_004E    0
B01001_005E    0
B01001_006E    0
B01001_007E    0
B01001_008E    0
B01001_009E    0
B01001_010E    0
B01001_011E    0
B01001_012E    0
B01001_013E    0
B01001_014E    0
B01001_015E    0
B01001_016E    0
dtype: int64


# Save processed dataset

In [48]:
# remove duplicate county name column if present
if "NAME.1" in final_df.columns:
    final_df = final_df.drop(columns=["NAME.1"])

# if duplicate NAME columns exist before saving, keep only the first one
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

# make sure county_fips is stored as 5-digit string
final_df["county_fips"] = final_df["county_fips"].astype(str).str.zfill(5)

print(final_df.shape)
display(final_df.head())

(3115, 162)


,GEO_ID,NAME,county_fips,B01001_001E,B01001_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_007E,...,B23025_006E,B23025_007E,B25001_001E,state,county_name,totalvotes,democrat_votes,republican_votes,party_winner,dem_vote_share
0,0500000US01001,"Autauga County, Alabama",01001,55639,27052,1727,2156,1674,1208,553,...,709,18335,23697,ALABAMA,AUTAUGA,27770,7503.0,19838.0,0,0.270184
1,0500000US01003,"Baldwin County, Alabama",01003,218289,105889,6082,5197,8289,4305,2406,...,321,73736,116747,ALABAMA,BALDWIN,109679,24578.0,83544.0,0,0.224090
2,0500000US01005,"Barbour County, Alabama",01005,25026,13156,660,741,721,467,261,...,0,11051,12057,ALABAMA,BARBOUR,10518,4816.0,5622.0,0,0.457882
3,0500000US01007,"Bibb County, Alabama",01007,22374,12022,626,693,621,569,323,...,0,9451,9237,ALABAMA,BIBB,9595,1986.0,7525.0,0,0.206983
4,0500000US01009,"Blount County, Alabama",01009,57755,28677,1844,1738,2034,1257,743,...,44,21950,24404,ALABAMA,BLOUNT,27588,2640.0,24711.0,0,0.095694


In [53]:
# keep only first duplicate columns if any
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

# force county_fips to string and preserve leading zeros
final_df["county_fips"] = final_df["county_fips"].astype(str)

# remove any accidental .0 if present
final_df["county_fips"] = final_df["county_fips"].str.replace(".0", "", regex=False)

# strip spaces
final_df["county_fips"] = final_df["county_fips"].str.strip()

# force 5 digits
final_df["county_fips"] = final_df["county_fips"].str.zfill(5)

display(final_df.head())

,GEO_ID,NAME,county_fips,B01001_001E,B01001_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_007E,...,B23025_006E,B23025_007E,B25001_001E,state,county_name,totalvotes,democrat_votes,republican_votes,party_winner,dem_vote_share
0,0500000US01001,"Autauga County, Alabama",01001,55639,27052,1727,2156,1674,1208,553,...,709,18335,23697,ALABAMA,AUTAUGA,27770,7503.0,19838.0,0,0.270184
1,0500000US01003,"Baldwin County, Alabama",01003,218289,105889,6082,5197,8289,4305,2406,...,321,73736,116747,ALABAMA,BALDWIN,109679,24578.0,83544.0,0,0.224090
2,0500000US01005,"Barbour County, Alabama",01005,25026,13156,660,741,721,467,261,...,0,11051,12057,ALABAMA,BARBOUR,10518,4816.0,5622.0,0,0.457882
3,0500000US01007,"Bibb County, Alabama",01007,22374,12022,626,693,621,569,323,...,0,9451,9237,ALABAMA,BIBB,9595,1986.0,7525.0,0,0.206983
4,0500000US01009,"Blount County, Alabama",01009,57755,28677,1844,1738,2034,1257,743,...,44,21950,24404,ALABAMA,BLOUNT,27588,2640.0,24711.0,0,0.095694


In [54]:
os.makedirs("../data/processed", exist_ok=True)

final_df.to_csv(
    "../data/processed/merged_county_dataset.csv",
    index=False
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


# Final verification

In [ ]:
saved_df = pd.read_csv(
    "../data/processed/merged_county_dataset.csv",
    dtype={"county_fips": str},
    low_memory=False
)

saved_df["county_fips"] = saved_df["county_fips"].astype(str).str.strip().str.zfill(5)

display(saved_df.head())

,GEO_ID,NAME,county_fips,B01001_001E,B01001_002E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_007E,...,B23025_006E,B23025_007E,B25001_001E,state,county_name,totalvotes,democrat_votes,republican_votes,party_winner,dem_vote_share
0,0500000US01001,"Autauga County, Alabama",01001,55639,27052,1727,2156,1674,1208,553,...,709,18335,23697,ALABAMA,AUTAUGA,27770,7503.0,19838.0,0,0.270184
1,0500000US01003,"Baldwin County, Alabama",01003,218289,105889,6082,5197,8289,4305,2406,...,321,73736,116747,ALABAMA,BALDWIN,109679,24578.0,83544.0,0,0.224090
2,0500000US01005,"Barbour County, Alabama",01005,25026,13156,660,741,721,467,261,...,0,11051,12057,ALABAMA,BARBOUR,10518,4816.0,5622.0,0,0.457882
3,0500000US01007,"Bibb County, Alabama",01007,22374,12022,626,693,621,569,323,...,0,9451,9237,ALABAMA,BIBB,9595,1986.0,7525.0,0,0.206983
4,0500000US01009,"Blount County, Alabama",01009,57755,28677,1844,1738,2034,1257,743,...,44,21950,24404,ALABAMA,BLOUNT,27588,2640.0,24711.0,0,0.095694


# Debugging

In [25]:
print("ACS county_fips sample:")
print(acs_merged["county_fips"].head(10).tolist())

print("\nMIT county_fips sample:")
print(mit_pivot["county_fips"].head(10).tolist())

print("\nACS county_fips dtype:", acs_merged["county_fips"].dtype)
print("MIT county_fips dtype:", mit_pivot["county_fips"].dtype)

ACS county_fips sample:
['01001', '01003', '01005', '01007', '01009', '01011', '01013', '01015', '01017', '01019']

MIT county_fips sample:
['00nan', '10001.0', '10003.0', '10005.0', '1001.0', '1003.0', '1005.0', '1007.0', '1009.0', '1011.0']

ACS county_fips dtype: object
MIT county_fips dtype: object


In [27]:
acs_fips = set(acs_merged["county_fips"])
mit_fips = set(mit_pivot["county_fips"])

common_fips = acs_fips.intersection(mit_fips)

print("ACS FIPS count:", len(acs_fips))
print("MIT FIPS count:", len(mit_fips))
print("Common FIPS count:", len(common_fips))

ACS FIPS count: 3221
MIT FIPS count: 3155
Common FIPS count: 0


In [28]:
mit_2020 = mit_df[mit_df["year"] == 2020].copy()
mit_2020 = mit_2020[mit_2020["party"].isin(["DEMOCRAT", "REPUBLICAN"])].copy()

mit_2020 = mit_2020[mit_2020["county_fips"].notna()].copy()

mit_2020["county_fips"] = (
    mit_2020["county_fips"]
    .astype(float)
    .astype(int)
    .astype(str)
    .str.zfill(5)
)